In [ ]:
# ============================================
# Moroccan Legal RAG - GPU Embedding Service
# Environment Setup
# ============================================

# Install dependencies with compatible versions
!pip install -q fastapi uvicorn==0.30.6 sentence-transformers nest-asyncio

# Download Cloudflare tunnel client
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

# Make it executable
!chmod +x cloudflared

# Required for running FastAPI inside Colab
import nest_asyncio
nest_asyncio.apply()

print("✓ Environment setup complete.")
print(f"  - fastapi, uvicorn==0.30.6, sentence-transformers installed")
print(f"  - cloudflared downloaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mcp 1.27.2 requires uvicorn>=0.31.1; sys_platform != "emscripten", but you have uvicorn 0.30.6 which is incompatible.
google-adk 1.29.0 requires uvicorn<1.0.0,>=0.34.0, but you have uvicorn 0.30.6 which is incompatible.
✓ Environment setup complete.
  - fastapi, uvicorn==0.30.6, sentence-transformers installed
  - cloudflared downloaded


In [ ]:
# ============================================
# Load BAAI/bge-m3 on GPU
# ============================================

from sentence_transformers import SentenceTransformer
import torch
import time

# Verify GPU
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ GPU not available.\n"
        "Go to: Runtime → Change runtime type → Select T4 GPU → Save\n"
        "Then re-run this cell."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
# print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print()

# Load model
print("Loading model...")
start = time.time()

model = SentenceTransformer("BAAI/bge-m3")
model = model.to("cuda")

elapsed = time.time() - start
print(f"✓ Model loaded in {elapsed:.1f}s")
print(f"  Device: {model.device}")
print(f"  Dimension: {model.get_sentence_embedding_dimension()}")


CUDA available: True
GPU: Tesla T4

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✓ Model loaded in 67.9s
  Device: cuda:0
  Dimension: 1024


/tmp/ipykernel_2067/2933266923.py:33: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"  Dimension: {model.get_sentence_embedding_dimension()}")


In [ ]:
# Warm-up
print("\nWarming up...")
warmup_start = time.time()
_ = model.encode(
    [
        "اختبار",              # Arabic
        "test juridique",      # French
        "legal test"           # English
    ],
    normalize_embeddings=True
)
warmup_elapsed = time.time() - warmup_start
print(f"✓ Warm-up complete in {warmup_elapsed:.2f}s")
print("\n✅ Model ready for requests.")


Warming up...
✓ Warm-up complete in 0.95s

✅ Model ready for requests.


In [ ]:
# ============================================
# FastAPI Application
# ============================================

from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np

app = FastAPI(
    title="Moroccan Legal RAG - Embedding Service",
    version="1.0.0"
)

# Request/Response models
class EmbedRequest(BaseModel):
    texts: list[str]
    batch_size: int = 32

class EmbedResponse(BaseModel):
    embeddings: list[list[float]]
    dimension: int
    count: int
    time_ms: float

# Routes
@app.get("/")
async def root():
    return {
        "message": "Moroccan Legal RAG Embedding Service",
        "docs": "/docs"
    }

@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model": "BAAI/bge-m3",
        "device": str(model.device),
        "dimension": model.get_sentence_embedding_dimension(),
        "gpu_name": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else "CPU"
        )
    }

@app.post("/embed", response_model=EmbedResponse)
async def embed(request: EmbedRequest):
    start = time.time()

    embeddings = model.encode(
        request.texts,
        batch_size=request.batch_size,
        normalize_embeddings=True,
        show_progress_bar=len(request.texts) > 50,
        convert_to_numpy=True
    )

    elapsed = (time.time() - start) * 1000

    return EmbedResponse(
        embeddings=embeddings.tolist(),
        dimension=int(embeddings.shape[1]),
        count=len(request.texts),
        time_ms=round(elapsed, 2)
    )

print("✓ FastAPI app created.")
print("  Routes: /  /health  /embed  /docs")

✓ FastAPI app created.
  Routes: /  /health  /embed  /docs


In [ ]:
# ============================================
# Start Cloudflare Tunnel + FastAPI Server
# ============================================

import subprocess
import threading
import uvicorn
import re
import time
import os

def start_tunnel():
    """Start cloudflared quick tunnel in background."""
    time.sleep(5)  # Wait for uvicorn to start

    process = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print(line.strip())

        # Extract tunnel URL
        match = re.search(
            r"https://[-a-zA-Z0-9]+\.trycloudflare\.com",
            line
        )

        if match:
            print()
            print("=" * 60)
            print("✅ PUBLIC TUNNEL URL:")
            print(f"   {match.group(0)}")
            print("=" * 60)
            print()
            print("Copy this into your local .env file:")
            print(f'EMBEDDING_SERVICE_URL={match.group(0)}')
            print()
            print("Test it:")
            print(f"  curl {match.group(0)}/health")
            print()

# Start tunnel in background thread
tunnel_thread = threading.Thread(target=start_tunnel, daemon=True)
tunnel_thread.start()

# Configure uvicorn to run in notebook
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

print("Starting FastAPI server...")
print("Tunnel URL will appear below once connected.")
print("-" * 60)

# Run server (this blocks the cell - expected behavior)
await server.serve()

INFO:     Started server process [2067]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Starting FastAPI server...
Tunnel URL will appear below once connected.
------------------------------------------------------------
2026-06-09T14:59:06Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-09T14:59:06Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-09T14:59:11Z INF +--------------------------------------------------------------------------------------------+
2026-06-09T14:59:11Z INF |  Your quick Tunnel has been created! Visi

/tmp/ipykernel_2067/1296929091.py:39: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "dimension": model.get_sentence_embedding_dimension(),


INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     102.97.178.206:0 - "GET /health HTTP/1.1" 200 

INFO:     Shutting down


2026-06-09T15:48:36Z INF Initiating graceful shutdown due to signal interrupt ...
